<a href="https://colab.research.google.com/github/FreddieLewin23/FreddieLewin23/blob/main/SIG_data_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/Data Exercise - Trade Data.csv')
df.head() # closed trade pool for 7 traders, 120,000 rows and spot
# data looks very tidy, no missing data, dates are in the correct format
df['Date'] = pd.to_datetime(df['Date'])
df.isna().sum()


,0
Date,0
TimeOfDay,0
Spot,0
BuyTrader,0
SellTrader,0


Assumptions of the data:
1. I start trading as soon as the data ends here. The strategy needs to be based off of what I have in the data without overfitting (hopefully)
2. Spot price has no pattern (ie Brownian motion etc) so strategy needs to be based off of the volume data only.
3. In live trading I will not be able to see the orders of who is trading (only see at the end of the day). If I do enter into a trade then I can see who I entered into the trade with. This will rule out intra-day strategies that rely on order flow from traders that seem profitbale in the training data. When I run backtesting on strategies I need to assume I can only see spot, time and the people I enter into trades with.

Constraints of trading:
1. up to 5 buys and 5 sells (so can do less) and need to end each day flat. Short selling allowed.
2. No trades last 30 minutes of trading (weird one here). If not flat going into last 30 minutes, I need to enter into the first trades that come up towards EoD.

Ideas and Plan of Action:
Firstly I want to have some idea of how the other traders trade with some simple analysis.

1. You can’t target flows by ID unless you “poke” the tape and see the counterparty. So strategies must either (i) probe to reveal IDs, or (ii) time-target windows where good traders are likely.
2.

In [14]:
df["Date"] = pd.to_datetime(df["Date"])
try:
    df["TimeOfDay"] = pd.to_datetime(df["TimeOfDay"]).dt.time
except Exception:
    pass
dt = pd.to_datetime(df["Date"].astype(str) + " " + df["TimeOfDay"].astype(str))
df["dt"] = dt
df = df.sort_values("dt").reset_index(drop=True)
price_col = None
for cand in ["Spot", "Price", "spot", "price", "Spot_Price"]:
    if cand in df.columns:
        price_col = cand
        break
buy_col = None
sell_col = None
for cand in df.columns:
    if "buy" in cand.lower():
        buy_col = cand
    if "sell" in cand.lower():
        sell_col = cand

# Derive day, time, and enforcement windows
df["day"] = df["dt"].dt.date
df["tod"] = df["dt"].dt.time

# Infer trading hours window (first and last time-of-day per day)
agg = df.groupby("day")["tod"].agg(["min", "max"]).reset_index()
start_t = agg["min"].mode().iloc[0]
end_t = agg["max"].mode().iloc[0]

cutoff_minutes = 30
# Compute cutoff time per day based on per-day max time
# We'll approximate by subtracting 30 minutes from each day's max and then take the mode
per_day_cutoff = (df.groupby("day")["dt"].max() - pd.Timedelta(minutes=cutoff_minutes)).dt.time
cutoff_t = per_day_cutoff.mode().iloc[0]


In [20]:
per_trader = {}
    # All trader IDs that appear anywhere
traders = pd.unique(pd.concat([df[buy_col], df[sell_col]], ignore_index=True))
traders = np.sort(traders)
# Trades where each trader appears on either side
rows = []
for tr in traders:
    buys = (df[buy_col] == tr).sum()
    sells = (df[sell_col] == tr).sum()
    total = buys + sells
    # Average time-of-day (in seconds) to see if someone is early/late concentrated
    sec = df.loc[(df[buy_col] == tr) | (df[sell_col] == tr), "dt"].dt.hour * 3600 + df.loc[(df[buy_col] == tr) | (df[sell_col] == tr), "dt"].dt.minute * 60
    avg_time_sec = sec.mean() if len(sec) else np.nan
    rows.append({"Trader": tr, "Buys": buys, "Sells": sells, "Total": total, "AvgTimeSec": avg_time_sec})
per_trader_df = pd.DataFrame(rows).sort_values("Total", ascending=False).reset_index(drop=True)

per_trader_df

,Trader,Buys,Sells,Total,AvgTimeSec
0,0,46188,56763,102951,45844.403648
1,4,21113,21113,42226,45595.765642
2,2,17878,17878,35756,45367.623336
3,5,16906,16906,33812,45617.651130
4,1,7018,7018,14036,45365.407524
5,3,10570,0,10570,45532.836329
6,6,575,570,1145,36268.873362


Looks like Trader 5 is long only, maybe something like a pension fund or retail trader just wanting long term exposure, where as trader 1, 2, 3, 4, 6 are flat overall so are market makers. trader 0 is overall short so may be something like a hedge fund who are taking positions on the direction of spot.

In [25]:
vwap = df.groupby("day")[price_col].mean().rename("DayVWAP") # volume weighted average prive
# Per-trader VWAP when they BUY vs when they SELL
t_buy_vwap = df.groupby("day").apply(lambda x: x.groupby(buy_col)[price_col].mean()).rename("BuyVWAP").reset_index()
t_sell_vwap = df.groupby("day").apply(lambda x: x.groupby(sell_col)[price_col].mean()).rename("SellVWAP").reset_index()
# Merge day-level VWAP into those
t_buy_vwap = t_buy_vwap.merge(vwap.reset_index(), on="day", how="left")
t_sell_vwap = t_sell_vwap.merge(vwap.reset_index(), on="day", how="left")
# Compute skews
t_buy_vwap["BuyMinusDayVWAP"] = t_buy_vwap["BuyVWAP"] - t_buy_vwap["DayVWAP"]
t_sell_vwap["SellMinusDayVWAP"] = t_sell_vwap["SellVWAP"] - t_sell_vwap["DayVWAP"]
# Aggregate across days
buy_edge = t_buy_vwap.groupby(t_buy_vwap.columns[1])["BuyMinusDayVWAP"].mean().rename("MeanBuySkew_vs_DayVWAP")
sell_edge = t_sell_vwap.groupby(t_sell_vwap.columns[1])["SellMinusDayVWAP"].mean().rename("MeanSellSkew_vs_DayVWAP")
# vwap_edge = pd.concat([buy_edge, sell_edge], axis=1).reset_index().rename(columns={vwap_edge.columns[0]: "Trader"})
buy_edge, sell_edge

/tmp/ipython-input-3950220791.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  t_buy_vwap = df.groupby("day").apply(lambda x: x.groupby(buy_col)[price_col].mean()).rename("BuyVWAP").reset_index()
/tmp/ipython-input-3950220791.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  t_sell_vwap = df.groupby("day").apply(lambda x: x.groupby(sell_col)[price_col].mean()).rename("SellVWAP").reset_index()


(BuyTrader
 0    0.017156
 1    0.004114
 2   -0.023379
 3    0.003429
 4   -0.013972
 5   -0.005139
 6   -0.085863
 Name: MeanBuySkew_vs_DayVWAP, dtype: float64,
 SellTrader
 0   -0.013828
 1   -0.005771
 2    0.024143
 4    0.009584
 5    0.008997
 6    0.025336
 Name: MeanSellSkew_vs_DayVWAP, dtype: float64)

In [26]:
# "Follow" vs "Fade" signal PnL proxy:
# If Trader X buys at time t and you follow (buy), close at the next trade at or after cutoff (forced EOD close).
# Similarly for sells (you sell then cover at or after cutoff).
def pnl_follow_fade(df, trader, mode="follow"):
    # mode="follow": do same side as trader; "fade": opposite side
    if price_col is None or buy_col is None or sell_col is None:
        return np.nan
    # Identify cutoff per day
    day_cut = df.groupby("day")["dt"].max() - pd.Timedelta(minutes=cutoff_minutes)
    day_cut = day_cut.to_dict()
    rets = []
    for d, day_df in df.groupby("day"):
        cut_dt = day_cut[d]
        # Entry trades for this trader during the open window (before cutoff)
        t_entries = day_df[(day_df["dt"] < cut_dt) & ((day_df[buy_col] == trader) | (day_df[sell_col] == trader))]
        if t_entries.empty:
            continue
        # Exit price = first print at or after cutoff (marketable close-out price proxy)
        exit_row = day_df[day_df["dt"] >= cut_dt].head(1)
        if exit_row.empty:
            continue
        exit_px = float(exit_row[price_col].iloc[0])
        # For each entry, compute PnL for 1-share
        for _, row in t_entries.iterrows():
            side = "buy" if row[buy_col] == trader else "sell"
            entry_px = float(row[price_col])
            if mode == "follow":
                # Do same side
                if side == "buy":
                    pnl = exit_px - entry_px
                else:
                    pnl = entry_px - exit_px
            else:  # fade
                # Do opposite side
                if side == "buy":
                    pnl = entry_px - exit_px
                else:
                    pnl = exit_px - entry_px
            rets.append(pnl)
    if len(rets) == 0:
        return np.nan
    return pd.Series(rets).mean(), pd.Series(rets).std(), len(rets)

In [27]:
ff_rows = []
for tr in per_trader_df["Trader"]:
    res_fol = pnl_follow_fade(df, tr, mode="follow")
    res_fade = pnl_follow_fade(df, tr, mode="fade")
    fol_mean, fol_std, fol_n = (res_fol if isinstance(res_fol, tuple) else (np.nan, np.nan, 0))
    fad_mean, fad_std, fad_n = (res_fade if isinstance(res_fade, tuple) else (np.nan, np.nan, 0))
    ff_rows.append({
        "Trader": tr,
        "Follow_meanPnL": fol_mean,
        "Follow_std": fol_std,
        "Follow_trades": fol_n,
        "Fade_meanPnL": fad_mean,
        "Fade_std": fad_std,
        "Fade_trades": fad_n
    })
ff_df = pd.DataFrame(ff_rows).sort_values("Follow_meanPnL", ascending=False).reset_index(drop=True)

In [31]:
ff_df
print("Per-trader activity overview", per_trader_df)
print("Per-trader 'follow vs fade' (close at daily cutoff)", ff_df)

summary = {
    "rows": len(df),
    "days": df["day"].nunique(),
    "start_day": str(df["day"].min()),
    "end_day": str(df["day"].max()),
    "start_time_mode": str(start_t),
    "end_time_mode": str(end_t),
    "cutoff_time_mode": str(cutoff_t),
    "price_col": price_col,
    "buy_col": buy_col,
    "sell_col": sell_col,
}
summary

Per-trader activity overview    Trader   Buys  Sells   Total    AvgTimeSec
0       0  46188  56763  102951  45844.403648
1       4  21113  21113   42226  45595.765642
2       2  17878  17878   35756  45367.623336
3       5  16906  16906   33812  45617.651130
4       1   7018   7018   14036  45365.407524
5       3  10570      0   10570  45532.836329
6       6    575    570    1145  36268.873362
Per-trader 'follow vs fade' (close at daily cutoff)    Trader  Follow_meanPnL  Follow_std  Follow_trades  Fade_meanPnL  Fade_std  \
0       6        0.099450    0.776424           1145     -0.099450  0.776424   
1       2        0.026677    0.573925          33052     -0.026677  0.573925   
2       4        0.013715    0.577965          38365     -0.013715  0.577965   
3       5        0.008185    0.572439          30703     -0.008185  0.572439   
4       3        0.002034    0.566220           9821     -0.002034  0.566220   
5       1       -0.009715    0.575653          12886      0.009715  0.5

{'rows': 120248,
 'days': 251,
 'start_day': '2022-01-03',
 'end_day': '2022-12-30',
 'start_time_mode': '09:30:08',
 'end_time_mode': '15:58:59',
 'cutoff_time_mode': '15:28:59',
 'price_col': 'Spot',
 'buy_col': 'BuyTrader',
 'sell_col': 'SellTrader'}

So what are we even looking at here, trader 6 stands out
Mean ≈ +0.10 on a per-trade basis vs σ around 0.78 means Sharpe of  0.13, which is enormous at the per-trade level given that each trade is big and the sample size is 1145 so big ish.

I want to follow Trader 6s trades (same-side) to the cutoff.
Traders 245 are consistent
Trader 3 is basically flat meaning neutral so maybe liquidity provision.
Traders 1 and 0 are consistently negative when followed. I dont want to follow them if I run a follwowing strategy

Fading them yields a mirror +0.010 to +0.018 edge per trade.
Trader 0 has large volume (102951 trades). If this was a proper market theyd have big influenc slippage wise if you were around them.